# OncoBind-ML — Random Forest Track
## Model A → Model B → Model C

Run from top to bottom.

- **Model A:** LIG + MUT + ENV
- **Model B:** Model A + PLIP
- **Model C:** Model B + VINA + P2

The same locked test rows are reused for fair A/B/C comparison.


## Random Forest progression

- **Model A:** clean RF baseline + basic tuning + OOB analysis
- **Model B:** interaction-aware RF + broader randomized tuning + refined local search
- **Model C:** full-feature RF + redundancy control + feature selection + strongest tuning


## Common setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

In [ ]:
from scipy.stats import pearsonr, spearmanr, randint, uniform

In [ ]:
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    RandomizedSearchCV,
    GridSearchCV,
    learning_curve,
    validation_curve
)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectFromModel

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend" / "app").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook inside the OncoBind-ML project folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEED = 42
TARGET = "DDG.EXP"
DATA_PATH = Path(os.environ.get("ONCOBIND_DATA_PATH", str(PROJECT_ROOT / "data" / "raw" / "MdrDB_Full_Scraped.csv")))
if not DATA_PATH.is_file() or DATA_PATH.stat().st_size == 0:
    raise FileNotFoundError(f"Upload the nonempty MdrDB dataset to {DATA_PATH}")
N_JOBS = int(os.environ.get("ONCOBIND_N_JOBS", "1"))
np.random.seed(SEED)


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.shape

In [ ]:
LIG = [c for c in df.columns if c.startswith("LIG.")]
MUT = [c for c in df.columns if c.startswith("MUT.")]
ENV = [c for c in df.columns if c.startswith("ENV.")]
PLIP = [c for c in df.columns if c.startswith("PLIP.")]
VINA = [c for c in df.columns if c == "vina" or (c.startswith("vina") and c[4:].isdigit())]
P2 = [c for c in df.columns if c.startswith(("P2.", "P2dl.", "P2dp."))]

In [ ]:
A_FEATURES = LIG + MUT + ENV
B_FEATURES = A_FEATURES + PLIP
C_FEATURES = B_FEATURES + VINA + P2

pd.Series({
    "A": len(A_FEATURES),
    "B": len(B_FEATURES),
    "C": len(C_FEATURES)
})

In [ ]:
y = pd.to_numeric(df[TARGET], errors="coerce")
valid = y.notna()

df_clean = df.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)

len(df_clean)

In [ ]:
all_idx = np.arange(len(df_clean))

dev_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=SEED
)

y_dev = y.iloc[dev_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

In [ ]:
def metrics(y_true, pred):
    return {
        "R2": r2_score(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MAE": mean_absolute_error(y_true, pred),
        "Pearson": pearsonr(y_true, pred).statistic,
        "Spearman": spearmanr(y_true, pred).statistic
    }

In [ ]:
def actual_vs_pred(y_true, pred, title):
    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(y_true, pred, s=10, alpha=0.30)
    lo = min(np.min(y_true), np.min(pred))
    hi = max(np.max(y_true), np.max(pred))
    plt.plot([lo, hi], [lo, hi], "--")
    plt.xlabel("Actual DDG.EXP")
    plt.ylabel("Predicted DDG.EXP")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def residual_plot(pred, residual, title):
    plt.figure(figsize=(6.5, 4))
    plt.scatter(pred, residual, s=10, alpha=0.30)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted DDG.EXP")
    plt.ylabel("Residual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# MODEL A — Random Forest
### LIG + MUT + ENV

Model A starts with a clean Random Forest baseline, then adds OOB evaluation and practical hyperparameter tuning.

In [ ]:
XA = df_clean[A_FEATURES].apply(pd.to_numeric, errors="coerce")
XA = XA.replace([np.inf, -np.inf], np.nan)
XA.shape

In [ ]:
XA_dev = XA.iloc[dev_idx].reset_index(drop=True)
XA_test = XA.iloc[test_idx].reset_index(drop=True)

In [ ]:
pd.Series({
    "Development rows": len(XA_dev),
    "Test rows": len(XA_test),
    "Features": XA_dev.shape[1]
})

In [ ]:
A_missing = XA_dev.isna().mean().sort_values(ascending=False)
A_missing.head(10).to_frame("Missing")

In [ ]:
tmp = A_missing.head(12).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(tmp.index, tmp.values)
plt.xlabel("Missing fraction")
plt.title("Model A missing values")
plt.tight_layout()
plt.show()

In [ ]:
A_constant = XA_dev.nunique(dropna=True)
A_constant = A_constant[A_constant <= 1].index.tolist()
A_constant

In [ ]:
A_KEEP = [c for c in XA_dev.columns if c not in A_constant]
len(A_KEEP)

In [ ]:
XA_dev = XA_dev[A_KEEP]
XA_test = XA_test[A_KEEP]

In [ ]:
A_imputer = SimpleImputer(strategy="median")

In [ ]:
XA_dev_imp = pd.DataFrame(
    A_imputer.fit_transform(XA_dev),
    columns=A_KEEP
)

In [ ]:
XA_test_imp = pd.DataFrame(
    A_imputer.transform(XA_test),
    columns=A_KEEP
)

In [ ]:
A_baseline = RandomForestRegressor(
    n_estimators=300,
    random_state=SEED,
    n_jobs=N_JOBS
)

In [ ]:
A_baseline.fit(XA_dev_imp, y_dev)

In [ ]:
A_base_train_pred = A_baseline.predict(XA_dev_imp)

In [ ]:
A_base_test_pred = A_baseline.predict(XA_test_imp)

In [ ]:
A_base_result = metrics(y_test, A_base_test_pred)
pd.Series(A_base_result).round(4)

In [ ]:
pd.DataFrame({
    "Split": ["Development", "Locked test"],
    "R2": [
        r2_score(y_dev, A_base_train_pred),
        r2_score(y_test, A_base_test_pred)
    ]
}).round(4)

In [ ]:
A_oob = RandomForestRegressor(
    n_estimators=400,
    bootstrap=True,
    oob_score=True,
    random_state=SEED,
    n_jobs=N_JOBS
)

In [ ]:
A_oob.fit(XA_dev_imp, y_dev)

In [ ]:
A_oob.oob_score_

In [ ]:
A_oob_residual = y_dev.to_numpy() - A_oob.oob_prediction_

In [ ]:
residual_plot(
    A_oob.oob_prediction_,
    A_oob_residual,
    "Model A OOB residuals"
)

In [ ]:
A_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        random_state=SEED,
        n_jobs=N_JOBS
    ))
])

In [ ]:
A_params = {
    "model__n_estimators": [250, 400, 600],
    "model__max_depth": [None, 12, 20, 30],
    "model__max_features": ["sqrt", 0.5, 0.8, 1.0],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__bootstrap": [True]
}

In [ ]:
A_search = RandomizedSearchCV(
    A_pipe,
    param_distributions=A_params,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=0
)

In [ ]:
A_search.fit(XA_dev, y_dev)

In [ ]:
A_search.best_params_

In [ ]:
-A_search.best_score_

In [ ]:
A_best = A_search.best_estimator_

In [ ]:
A_test_pred = A_best.predict(XA_test)

In [ ]:
A_result = metrics(y_test, A_test_pred)
pd.Series(A_result).round(4)

In [ ]:
actual_vs_pred(
    y_test,
    A_test_pred,
    "Model A — tuned Random Forest"
)

In [ ]:
A_residual = y_test.to_numpy() - A_test_pred

In [ ]:
residual_plot(
    A_test_pred,
    A_residual,
    "Model A residuals"
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(A_residual, bins=50)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Model A residual distribution")
plt.tight_layout()
plt.show()

In [ ]:
A_core = A_best.named_steps["model"]

A_importance = pd.DataFrame({
    "Feature": A_KEEP,
    "Importance": A_core.feature_importances_
}).sort_values("Importance", ascending=False)

A_importance.head(15).round(4)

In [ ]:
tmp = A_importance.head(15).sort_values("Importance")
plt.figure(figsize=(8, 5))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Impurity importance")
plt.title("Model A feature importance")
plt.tight_layout()
plt.show()

In [ ]:
A_perm = permutation_importance(
    A_best,
    XA_test,
    y_test,
    n_repeats=5,
    random_state=SEED,
    scoring="r2",
    n_jobs=N_JOBS
)

In [ ]:
A_perm_df = pd.DataFrame({
    "Feature": A_KEEP,
    "Importance": A_perm.importances_mean
}).sort_values("Importance", ascending=False)

A_perm_df.head(15).round(4)

In [ ]:
tmp = A_perm_df.head(15).sort_values("Importance")
plt.figure(figsize=(8, 5))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Permutation importance")
plt.title("Model A permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
A_depths = [tree.tree_.max_depth for tree in A_core.estimators_]
pd.Series(A_depths).describe().round(2)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(A_depths, bins=20)
plt.xlabel("Tree depth")
plt.ylabel("Trees")
plt.title("Model A tree-depth distribution")
plt.tight_layout()
plt.show()

In [ ]:
A_leaves = [tree.tree_.n_leaves for tree in A_core.estimators_]
pd.Series(A_leaves).describe().round(2)

In [ ]:
A_error = pd.DataFrame({
    "Actual": y_test,
    "Predicted": A_test_pred,
    "Residual": A_residual
})
A_error["Absolute error"] = A_error["Residual"].abs()
A_error.sort_values("Absolute error", ascending=False).head(12).round(4)

In [ ]:
A_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

In [ ]:
A_cv_score = cross_validate(
    A_best,
    XA_dev,
    y_dev,
    cv=A_cv,
    scoring={
        "R2": "r2",
        "RMSE": "neg_root_mean_squared_error"
    },
    n_jobs=N_JOBS
)

In [ ]:
pd.Series({
    "CV R2": A_cv_score["test_R2"].mean(),
    "CV RMSE": -A_cv_score["test_RMSE"].mean()
}).round(4)

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    A_best,
    XA_dev,
    y_dev,
    cv=3,
    scoring="r2",
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=N_JOBS
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Train")
plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="CV")
plt.xlabel("Training rows")
plt.ylabel("R²")
plt.title("Model A learning curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
A_predictions = pd.DataFrame({
    "SAMPLE_ID": df_clean.iloc[test_idx]["SAMPLE_ID"].values,
    "Actual": y_test,
    "Predicted": A_test_pred,
    "Residual": A_residual
})
A_predictions.to_csv("RF_Model_A_predictions.csv", index=False)

In [ ]:
joblib.dump(
    {
        "model": A_best,
        "features": A_KEEP,
        "best_params": A_search.best_params_
    },
    "RF_Model_A.joblib"
)

In [ ]:
A_final = pd.DataFrame([{
    "Model": "A",
    "Method": "Tuned Random Forest",
    "Raw features": len(A_FEATURES),
    **A_result
}])
A_final.round(4)

# MODEL B — Interaction-Aware Random Forest
### Model A + PLIP

Model B adds PLIP interaction descriptors, interaction summaries, wider tuning, OOB checking and a local refinement search.

In [ ]:
XB = df_clean[B_FEATURES].apply(pd.to_numeric, errors="coerce")
XB = XB.replace([np.inf, -np.inf], np.nan)
XB.shape

In [ ]:
XB_dev = XB.iloc[dev_idx].reset_index(drop=True)
XB_test = XB.iloc[test_idx].reset_index(drop=True)

In [ ]:
XB_dev[PLIP].describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
B_plip_nonzero = (XB_dev[PLIP] > 0).mean().sort_values()
B_plip_nonzero.to_frame("Non-zero fraction")

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(B_plip_nonzero.index, B_plip_nonzero.values)
plt.xlabel("Non-zero fraction")
plt.title("Model B PLIP activity")
plt.tight_layout()
plt.show()

In [ ]:
def add_B_features(X):
    X = X.copy()
    X["FE.PLIPTotal"] = X[PLIP].sum(axis=1, min_count=1)
    X["FE.PLIPMean"] = X[PLIP].mean(axis=1)
    X["FE.PLIPStd"] = X[PLIP].std(axis=1)
    X["FE.PLIPActive"] = (X[PLIP] > 0).sum(axis=1)

    if "LIG.MolWt" in X.columns:
        X["FE.PLIPPerMolWt"] = (
            X["FE.PLIPTotal"] /
            X["LIG.MolWt"].replace(0, np.nan)
        )

    return X.replace([np.inf, -np.inf], np.nan)

In [ ]:
XB_dev_fe = add_B_features(XB_dev)
XB_test_fe = add_B_features(XB_test)
XB_dev_fe.shape

In [ ]:
B_missing = XB_dev_fe.isna().mean().sort_values(ascending=False)
B_missing.head(10).to_frame("Missing")

In [ ]:
B_constant = XB_dev_fe.nunique(dropna=True)
B_constant = B_constant[B_constant <= 1].index.tolist()
B_constant

In [ ]:
B_KEEP = [c for c in XB_dev_fe.columns if c not in B_constant]
XB_dev_fe = XB_dev_fe[B_KEEP]
XB_test_fe = XB_test_fe[B_KEEP]
len(B_KEEP)

In [ ]:
B_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        bootstrap=True,
        oob_score=True,
        random_state=SEED,
        n_jobs=N_JOBS
    ))
])

In [ ]:
B_params = {
    "model__n_estimators": randint(350, 950),
    "model__max_depth": [None, 12, 16, 20, 24, 30, 40],
    "model__max_features": uniform(0.30, 0.65),
    "model__min_samples_split": randint(2, 15),
    "model__min_samples_leaf": randint(1, 7),
    "model__max_samples": [None, 0.7, 0.85, 0.95]
}

In [ ]:
B_search = RandomizedSearchCV(
    B_pipe,
    param_distributions=B_params,
    n_iter=28,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=0
)

In [ ]:
B_search.fit(XB_dev_fe, y_dev)

In [ ]:
B_search.best_params_

In [ ]:
-B_search.best_score_

In [ ]:
B_best_random = B_search.best_estimator_

In [ ]:
B_core_random = B_best_random.named_steps["model"]
B_core_random.oob_score_

In [ ]:
B_random_pred = B_best_random.predict(XB_test_fe)

In [ ]:
B_random_result = metrics(y_test, B_random_pred)
pd.Series(B_random_result).round(4)

In [ ]:
best = B_search.best_params_

In [ ]:
best_depth = best["model__max_depth"]
best_leaf = best["model__min_samples_leaf"]
best_split = best["model__min_samples_split"]
best_features = best["model__max_features"]

In [ ]:
depth_grid = [best_depth] if best_depth is None else sorted(
    set([max(4, best_depth - 4), best_depth, best_depth + 4])
)

leaf_grid = sorted(set([
    max(1, best_leaf - 1),
    best_leaf,
    best_leaf + 1
]))

In [ ]:
B_refine_grid = {
    "model__n_estimators": [500, 700, 900],
    "model__max_depth": depth_grid,
    "model__min_samples_leaf": leaf_grid,
    "model__min_samples_split": sorted(set([
        max(2, best_split - 2),
        best_split,
        best_split + 2
    ])),
    "model__max_features": [best_features]
}

In [ ]:
B_refine = GridSearchCV(
    B_pipe,
    B_refine_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=N_JOBS,
    verbose=0
)

In [ ]:
B_refine.fit(XB_dev_fe, y_dev)

In [ ]:
B_refine.best_params_

In [ ]:
-B_refine.best_score_

In [ ]:
B_best = B_refine.best_estimator_

In [ ]:
B_core = B_best.named_steps["model"]
B_core.oob_score_

In [ ]:
B_test_pred = B_best.predict(XB_test_fe)

In [ ]:
B_result = metrics(y_test, B_test_pred)
pd.Series(B_result).round(4)

In [ ]:
actual_vs_pred(
    y_test,
    B_test_pred,
    "Model B — tuned Random Forest"
)

In [ ]:
B_residual = y_test.to_numpy() - B_test_pred

In [ ]:
residual_plot(
    B_test_pred,
    B_residual,
    "Model B residuals"
)

In [ ]:
B_importance = pd.DataFrame({
    "Feature": B_KEEP,
    "Importance": B_core.feature_importances_
}).sort_values("Importance", ascending=False)

B_importance.head(20).round(4)

In [ ]:
tmp = B_importance.head(20).sort_values("Importance")
plt.figure(figsize=(8, 6))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Importance")
plt.title("Model B feature importance")
plt.tight_layout()
plt.show()

In [ ]:
B_plip_importance = B_importance[
    B_importance["Feature"].str.startswith(("PLIP.", "FE.PLIP"))
]
B_plip_importance.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(
    B_plip_importance["Feature"],
    B_plip_importance["Importance"]
)
plt.xlabel("Importance")
plt.title("Model B PLIP importance")
plt.tight_layout()
plt.show()

In [ ]:
B_perm = permutation_importance(
    B_best,
    XB_test_fe,
    y_test,
    n_repeats=5,
    random_state=SEED,
    scoring="r2",
    n_jobs=N_JOBS
)

In [ ]:
B_perm_df = pd.DataFrame({
    "Feature": B_KEEP,
    "Importance": B_perm.importances_mean
}).sort_values("Importance", ascending=False)

B_perm_df.head(20).round(4)

In [ ]:
tmp = B_perm_df.head(20).sort_values("Importance")
plt.figure(figsize=(8, 6))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Permutation importance")
plt.title("Model B permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
B_depths = [tree.tree_.max_depth for tree in B_core.estimators_]
pd.Series(B_depths).describe().round(2)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(B_depths, bins=20)
plt.xlabel("Tree depth")
plt.ylabel("Trees")
plt.title("Model B tree-depth distribution")
plt.tight_layout()
plt.show()

In [ ]:
B_oob_pred = B_core.oob_prediction_
B_oob_residual = y_dev.to_numpy() - B_oob_pred

In [ ]:
actual_vs_pred(
    y_dev,
    B_oob_pred,
    "Model B OOB prediction"
)

In [ ]:
residual_plot(
    B_oob_pred,
    B_oob_residual,
    "Model B OOB residuals"
)

In [ ]:
B_cv_score = cross_validate(
    B_best,
    XB_dev_fe,
    y_dev,
    cv=3,
    scoring={
        "R2": "r2",
        "RMSE": "neg_root_mean_squared_error"
    },
    n_jobs=N_JOBS
)

In [ ]:
pd.Series({
    "CV R2": B_cv_score["test_R2"].mean(),
    "CV RMSE": -B_cv_score["test_RMSE"].mean()
}).round(4)

In [ ]:
param_range = [100, 250, 400, 600, 800]
train_s, val_s = validation_curve(
    B_best,
    XB_dev_fe,
    y_dev,
    param_name="model__n_estimators",
    param_range=param_range,
    cv=3,
    scoring="r2",
    n_jobs=N_JOBS
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(param_range, train_s.mean(axis=1), marker="o", label="Train")
plt.plot(param_range, val_s.mean(axis=1), marker="o", label="CV")
plt.xlabel("Number of trees")
plt.ylabel("R²")
plt.title("Model B n_estimators curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
B_error = pd.DataFrame({
    "Actual": y_test,
    "Predicted": B_test_pred,
    "Residual": B_residual
})
B_error["Absolute error"] = B_error["Residual"].abs()
B_error.sort_values("Absolute error", ascending=False).head(12).round(4)

In [ ]:
AB_results = pd.DataFrame([
    {"Model": "A", **A_result},
    {"Model": "B", **B_result}
])
AB_results.round(4)

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(AB_results["Model"], AB_results["R2"])
plt.ylabel("Locked test R²")
plt.title("Random Forest A vs B")
plt.tight_layout()
plt.show()

In [ ]:
B_predictions = pd.DataFrame({
    "SAMPLE_ID": df_clean.iloc[test_idx]["SAMPLE_ID"].values,
    "Actual": y_test,
    "Predicted": B_test_pred,
    "Residual": B_residual
})
B_predictions.to_csv("RF_Model_B_predictions.csv", index=False)

In [ ]:
joblib.dump(
    {
        "model": B_best,
        "features": B_KEEP,
        "random_search": B_search.best_params_,
        "refined_search": B_refine.best_params_
    },
    "RF_Model_B.joblib"
)

In [ ]:
B_final = pd.DataFrame([{
    "Model": "B",
    "Method": "Interaction-aware tuned RF",
    "Raw features": len(B_FEATURES),
    **B_result
}])
B_final.round(4)

# MODEL C — Full Random Forest
### Model B + VINA + P2

Model C uses all biochemical/docking features, engineered summaries, feature selection, wider tuning and local refinement.

In [ ]:
XC = df_clean[C_FEATURES].apply(pd.to_numeric, errors="coerce")
XC = XC.replace([np.inf, -np.inf], np.nan)
XC.shape

In [ ]:
XC_dev = XC.iloc[dev_idx].reset_index(drop=True)
XC_test = XC.iloc[test_idx].reset_index(drop=True)

In [ ]:
C_family_missing = pd.Series({
    "LIG": XC_dev[LIG].isna().mean().mean(),
    "MUT": XC_dev[MUT].isna().mean().mean(),
    "ENV": XC_dev[ENV].isna().mean().mean(),
    "PLIP": XC_dev[PLIP].isna().mean().mean(),
    "VINA": XC_dev[VINA].isna().mean().mean(),
    "P2": XC_dev[P2].isna().mean().mean()
})
C_family_missing.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(C_family_missing.index, C_family_missing.values)
plt.ylabel("Missing fraction")
plt.title("Model C missingness by family")
plt.tight_layout()
plt.show()

In [ ]:
def add_C_features(X):
    X = X.copy()

    X["FE.PLIPTotal"] = X[PLIP].sum(axis=1, min_count=1)

    X["FE.VinaMean"] = X[VINA].mean(axis=1)
    X["FE.VinaStd"] = X[VINA].std(axis=1)
    X["FE.VinaMin"] = X[VINA].min(axis=1)
    X["FE.VinaMax"] = X[VINA].max(axis=1)
    X["FE.VinaRange"] = X["FE.VinaMax"] - X["FE.VinaMin"]
    X["FE.VinaMedian"] = X[VINA].median(axis=1)

    p2 = [c for c in P2 if c.startswith("P2.")]
    p2dl = [c for c in P2 if c.startswith("P2dl.")]
    p2dp = [c for c in P2 if c.startswith("P2dp.")]

    if p2:
        X["FE.P2Total"] = X[p2].sum(axis=1, min_count=1)

    if p2dl:
        X["FE.P2dlTotal"] = X[p2dl].sum(axis=1, min_count=1)

    if p2dp:
        X["FE.P2dpTotal"] = X[p2dp].sum(axis=1, min_count=1)

    if p2dl and p2dp:
        X["FE.P2Contrast"] = X["FE.P2dlTotal"] - X["FE.P2dpTotal"]

    return X.replace([np.inf, -np.inf], np.nan)

In [ ]:
XC_dev_fe = add_C_features(XC_dev)
XC_test_fe = add_C_features(XC_test)
XC_dev_fe.shape

In [ ]:
C_constant = XC_dev_fe.nunique(dropna=True)
C_constant = C_constant[C_constant <= 1].index.tolist()
C_constant

In [ ]:
C_KEEP = [c for c in XC_dev_fe.columns if c not in C_constant]
XC_dev_fe = XC_dev_fe[C_KEEP]
XC_test_fe = XC_test_fe[C_KEEP]
len(C_KEEP)

In [ ]:
C_imputer = SimpleImputer(strategy="median")
XC_dev_imp = pd.DataFrame(
    C_imputer.fit_transform(XC_dev_fe),
    columns=C_KEEP
)
XC_test_imp = pd.DataFrame(
    C_imputer.transform(XC_test_fe),
    columns=C_KEEP
)

In [ ]:
C_probe = RandomForestRegressor(
    n_estimators=350,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=N_JOBS
)

In [ ]:
C_probe.fit(XC_dev_imp, y_dev)

In [ ]:
C_probe_imp = pd.DataFrame({
    "Feature": C_KEEP,
    "Importance": C_probe.feature_importances_
}).sort_values("Importance", ascending=False)
C_probe_imp.head(20).round(4)

In [ ]:
tmp = C_probe_imp.head(20).sort_values("Importance")
plt.figure(figsize=(8, 6))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Importance")
plt.title("Model C probe importance")
plt.tight_layout()
plt.show()

In [ ]:
C_selector = SelectFromModel(
    C_probe,
    threshold="median",
    prefit=True
)

In [ ]:
C_SELECTED = np.array(C_KEEP)[C_selector.get_support()].tolist()
len(C_SELECTED)

In [ ]:
XC_dev_sel = XC_dev_imp[C_SELECTED]
XC_test_sel = XC_test_imp[C_SELECTED]

In [ ]:
C_family_selected = pd.Series({
    "LIG": len([c for c in C_SELECTED if c.startswith("LIG.")]),
    "MUT": len([c for c in C_SELECTED if c.startswith("MUT.")]),
    "ENV": len([c for c in C_SELECTED if c.startswith("ENV.")]),
    "PLIP": len([c for c in C_SELECTED if c.startswith("PLIP.")]),
    "VINA": len([c for c in C_SELECTED if c == "vina" or c.startswith("vina")]),
    "P2": len([c for c in C_SELECTED if c.startswith(("P2.", "P2dl.", "P2dp."))]),
    "Engineered": len([c for c in C_SELECTED if c.startswith("FE.")])
})
C_family_selected

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(C_family_selected.index, C_family_selected.values)
plt.ylabel("Selected features")
plt.title("Model C selected feature families")
plt.tight_layout()
plt.show()

In [ ]:
C_rf = RandomForestRegressor(
    bootstrap=True,
    oob_score=True,
    random_state=SEED,
    n_jobs=N_JOBS
)

In [ ]:
C_params = {
    "n_estimators": randint(450, 1200),
    "max_depth": [None, 14, 18, 22, 26, 32, 40],
    "max_features": uniform(0.25, 0.70),
    "min_samples_split": randint(2, 18),
    "min_samples_leaf": randint(1, 8),
    "max_samples": [None, 0.65, 0.75, 0.85, 0.95],
    "criterion": ["squared_error", "friedman_mse"]
}

In [ ]:
C_search = RandomizedSearchCV(
    C_rf,
    param_distributions=C_params,
    n_iter=32,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=0
)

In [ ]:
C_search.fit(XC_dev_sel, y_dev)

In [ ]:
C_search.best_params_

In [ ]:
-C_search.best_score_

In [ ]:
C_random_best = C_search.best_estimator_

In [ ]:
C_random_best.oob_score_

In [ ]:
C_random_pred = C_random_best.predict(XC_test_sel)
C_random_result = metrics(y_test, C_random_pred)
pd.Series(C_random_result).round(4)

In [ ]:
cbest = C_search.best_params_

In [ ]:
c_depth = cbest["max_depth"]
c_leaf = cbest["min_samples_leaf"]
c_split = cbest["min_samples_split"]
c_feat = cbest["max_features"]
c_crit = cbest["criterion"]

In [ ]:
c_depth_grid = [c_depth] if c_depth is None else sorted(
    set([max(6, c_depth - 4), c_depth, c_depth + 4])
)

In [ ]:
C_refine_grid = {
    "n_estimators": [650, 850, 1050],
    "max_depth": c_depth_grid,
    "min_samples_leaf": sorted(set([
        max(1, c_leaf - 1),
        c_leaf,
        c_leaf + 1
    ])),
    "min_samples_split": sorted(set([
        max(2, c_split - 2),
        c_split,
        c_split + 2
    ])),
    "max_features": [c_feat],
    "criterion": [c_crit]
}

In [ ]:
C_refine = GridSearchCV(
    RandomForestRegressor(
        bootstrap=True,
        oob_score=True,
        random_state=SEED,
        n_jobs=N_JOBS
    ),
    C_refine_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=N_JOBS,
    verbose=0
)

In [ ]:
C_refine.fit(XC_dev_sel, y_dev)

In [ ]:
C_refine.best_params_

In [ ]:
-C_refine.best_score_

In [ ]:
C_best = C_refine.best_estimator_

In [ ]:
C_best.oob_score_

In [ ]:
C_test_pred = C_best.predict(XC_test_sel)

In [ ]:
C_result = metrics(y_test, C_test_pred)
pd.Series(C_result).round(4)

In [ ]:
actual_vs_pred(
    y_test,
    C_test_pred,
    "Model C — tuned Random Forest"
)

In [ ]:
C_residual = y_test.to_numpy() - C_test_pred

In [ ]:
residual_plot(
    C_test_pred,
    C_residual,
    "Model C residuals"
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(C_residual, bins=50)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Model C residual distribution")
plt.tight_layout()
plt.show()

In [ ]:
C_importance = pd.DataFrame({
    "Feature": C_SELECTED,
    "Importance": C_best.feature_importances_
}).sort_values("Importance", ascending=False)

C_importance.head(25).round(4)

In [ ]:
tmp = C_importance.head(25).sort_values("Importance")
plt.figure(figsize=(9, 7))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Importance")
plt.title("Model C feature importance")
plt.tight_layout()
plt.show()

In [ ]:
def feature_family(name):
    if name.startswith("LIG."): return "LIG"
    if name.startswith("MUT."): return "MUT"
    if name.startswith("ENV."): return "ENV"
    if name.startswith("PLIP."): return "PLIP"
    if name == "vina" or name.startswith("vina"): return "VINA"
    if name.startswith(("P2.", "P2dl.", "P2dp.")): return "P2"
    if name.startswith("FE."): return "Engineered"
    return "Other"

In [ ]:
C_importance["Family"] = C_importance["Feature"].map(feature_family)
C_family_importance = C_importance.groupby("Family")["Importance"].sum().sort_values(ascending=False)
C_family_importance.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(C_family_importance.index, C_family_importance.values)
plt.ylabel("Total RF importance")
plt.title("Model C importance by family")
plt.tight_layout()
plt.show()

In [ ]:
C_perm = permutation_importance(
    C_best,
    XC_test_sel,
    y_test,
    n_repeats=5,
    random_state=SEED,
    scoring="r2",
    n_jobs=N_JOBS
)

In [ ]:
C_perm_df = pd.DataFrame({
    "Feature": C_SELECTED,
    "Importance": C_perm.importances_mean
}).sort_values("Importance", ascending=False)

C_perm_df.head(20).round(4)

In [ ]:
tmp = C_perm_df.head(20).sort_values("Importance")
plt.figure(figsize=(8, 6))
plt.barh(tmp["Feature"], tmp["Importance"])
plt.xlabel("Permutation importance")
plt.title("Model C permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
C_depths = [tree.tree_.max_depth for tree in C_best.estimators_]
pd.Series(C_depths).describe().round(2)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(C_depths, bins=20)
plt.xlabel("Tree depth")
plt.ylabel("Trees")
plt.title("Model C tree-depth distribution")
plt.tight_layout()
plt.show()

In [ ]:
C_leaves = [tree.tree_.n_leaves for tree in C_best.estimators_]
pd.Series(C_leaves).describe().round(2)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(C_leaves, bins=20)
plt.xlabel("Leaves per tree")
plt.ylabel("Trees")
plt.title("Model C leaf distribution")
plt.tight_layout()
plt.show()

In [ ]:
C_oob_pred = C_best.oob_prediction_
C_oob_result = metrics(y_dev, C_oob_pred)
pd.Series(C_oob_result).round(4)

In [ ]:
actual_vs_pred(
    y_dev,
    C_oob_pred,
    "Model C OOB prediction"
)

In [ ]:
C_cv_score = cross_validate(
    C_best,
    XC_dev_sel,
    y_dev,
    cv=3,
    scoring={
        "R2": "r2",
        "RMSE": "neg_root_mean_squared_error",
        "MAE": "neg_mean_absolute_error"
    },
    n_jobs=N_JOBS
)

In [ ]:
pd.Series({
    "CV R2": C_cv_score["test_R2"].mean(),
    "CV RMSE": -C_cv_score["test_RMSE"].mean(),
    "CV MAE": -C_cv_score["test_MAE"].mean()
}).round(4)

In [ ]:
C_train_pred = C_best.predict(XC_dev_sel)
pd.DataFrame({
    "Split": ["Development", "OOB", "Locked test"],
    "R2": [
        r2_score(y_dev, C_train_pred),
        C_best.oob_score_,
        r2_score(y_test, C_test_pred)
    ]
}).round(4)

In [ ]:
C_error = pd.DataFrame({
    "Actual": y_test,
    "Predicted": C_test_pred,
    "Residual": C_residual
})
C_error["Absolute error"] = C_error["Residual"].abs()
C_error.sort_values("Absolute error", ascending=False).head(15).round(4)

In [ ]:
C_bins = pd.qcut(y_test, q=5, duplicates="drop")
C_bin_result = (
    pd.DataFrame({
        "Actual": y_test,
        "Predicted": C_test_pred,
        "Bin": C_bins
    })
    .groupby("Bin", observed=True)
    .apply(lambda z: pd.Series({
        "RMSE": np.sqrt(mean_squared_error(z["Actual"], z["Predicted"])),
        "MAE": mean_absolute_error(z["Actual"], z["Predicted"])
    }))
)
C_bin_result.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    range(len(C_bin_result)),
    C_bin_result["RMSE"],
    marker="o"
)
plt.xlabel("Target quintile")
plt.ylabel("RMSE")
plt.title("Model C error by target range")
plt.tight_layout()
plt.show()

In [ ]:
ABC_results = pd.DataFrame([
    {"Model": "A", "Method": "Tuned RF", **A_result},
    {"Model": "B", "Method": "Interaction RF", **B_result},
    {"Model": "C", "Method": "Full selected RF", **C_result}
])
ABC_results.round(4)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(ABC_results["Model"], ABC_results["R2"])
plt.ylabel("Locked test R²")
plt.title("Random Forest track — A vs B vs C")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(ABC_results["Model"], ABC_results["RMSE"])
plt.ylabel("Locked test RMSE")
plt.title("Random Forest track — RMSE")
plt.tight_layout()
plt.show()

In [ ]:
ABC_change = pd.DataFrame({
    "Transition": ["A → B", "B → C", "A → C"],
    "Delta R2": [
        B_result["R2"] - A_result["R2"],
        C_result["R2"] - B_result["R2"],
        C_result["R2"] - A_result["R2"]
    ],
    "Delta RMSE": [
        B_result["RMSE"] - A_result["RMSE"],
        C_result["RMSE"] - B_result["RMSE"],
        C_result["RMSE"] - A_result["RMSE"]
    ]
})
ABC_change.round(4)

In [ ]:
C_predictions = pd.DataFrame({
    "SAMPLE_ID": df_clean.iloc[test_idx]["SAMPLE_ID"].values,
    "Actual": y_test,
    "Predicted": C_test_pred,
    "Residual": C_residual
})
C_predictions.to_csv("RF_Model_C_predictions.csv", index=False)

In [ ]:
ABC_results.to_csv(
    "RF_Model_A_B_C_results.csv",
    index=False
)

In [ ]:
joblib.dump(
    {
        "model": C_best,
        "selected_features": C_SELECTED,
        "best_params": C_refine.best_params_,
        "seed": SEED
    },
    "RF_Model_C.joblib"
)

In [ ]:
final_table = ABC_results.copy()
final_table["Raw features"] = [
    len(A_FEATURES),
    len(B_FEATURES),
    len(C_FEATURES)
]

final_table[
    [
        "Model",
        "Method",
        "Raw features",
        "R2",
        "RMSE",
        "MAE",
        "Pearson",
        "Spearman"
    ]
].round(4)

# Final interpretation

- **Model A:** baseline forest behavior and basic tuning.
- **Model B:** interaction-aware representation with PLIP, OOB validation and refined tuning.
- **Model C:** full biochemical/docking representation with RF-based feature selection and strongest tuning.

Use development CV and OOB results for model decisions. Use the locked test only for final comparison.


## Export to the application
Run after all training cells. This verifies predictions against the locked test before exporting fitted preprocessing, models, feature lists, metrics, and plots.

In [ ]:
from backend.training.export import export_track
export_track(globals(), "random_forest", PROJECT_ROOT)
